# Salary Equity Analysis

## Project Overview
Analyze compensation data across job levels, tenure, gender, and department to investigate salary distribution and possible disparities.

## Learning Objectives
- Analyze salary distributions across demographic and organizational dimensions
- Calculate pay gaps and compare median compensation across groups
- Identify potential equity concerns using ratio-based and residual-based approaches
- Visualize compensation patterns for HR decision support

## Problem Statement
HR and leadership want to ensure fair compensation practices. They need analytical evidence to understand whether salary differences exist across gender, department, and level — and whether those differences persist after controlling for role and tenure.

## Why This Project Matters
Salary inequity damages morale, increases attrition, and carries legal risk. Proactive pay equity analysis enables organizations to identify and correct disparities before they escalate.

## Dataset Overview
Synthetic compensation dataset: ~4,000 employees with salary, level, tenure, gender, department, and performance data.

## Dataset Source and License Notes
- Synthetic data
- No license restrictions

## Environment Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', palette='Set2')
np.random.seed(42)
print('Imports OK')

## Configuration / Constants

In [ ]:
import os
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print(f'Save dir: {SAVE_DIR}')

## Dataset Download or Loading

In [ ]:
np.random.seed(42)
n = 4000
departments = np.random.choice(['Engineering', 'Sales', 'Marketing', 'Finance', 'Operations', 'HR'],
                                 n, p=[0.25, 0.20, 0.15, 0.15, 0.15, 0.10])
levels = np.random.choice(['Junior', 'Mid', 'Senior', 'Lead', 'Director'], n, p=[0.25, 0.30, 0.25, 0.13, 0.07])
gender = np.random.choice(['Male', 'Female', 'Non-Binary'], n, p=[0.50, 0.46, 0.04])
tenure_years = np.clip(np.random.exponential(4, n), 0.5, 25).round(1)
perf = np.random.choice([1, 2, 3, 4, 5], n, p=[0.05, 0.12, 0.38, 0.32, 0.13])
education = np.random.choice(['High School', 'Bachelors', 'Masters', 'PhD'], n, p=[0.08, 0.45, 0.35, 0.12])

# Base salary by level
level_base = {'Junior': 55000, 'Mid': 75000, 'Senior': 100000, 'Lead': 125000, 'Director': 160000}
dept_mult = {'Engineering': 1.15, 'Sales': 1.0, 'Marketing': 0.95, 'Finance': 1.08, 'Operations': 0.92, 'HR': 0.90}
edu_add = {'High School': -5000, 'Bachelors': 0, 'Masters': 8000, 'PhD': 15000}

salary = np.array([level_base[l] * dept_mult[d] + edu_add[e] + tenure_years[i] * 1500 + perf[i] * 2000
                    for i, (l, d, e) in enumerate(zip(levels, departments, education))])
# Add small gender gap for analysis purposes (intentional in synthetic data)
gender_adj = np.where(gender == 'Female', -2500, np.where(gender == 'Non-Binary', -1500, 0))
salary = salary + gender_adj + np.random.normal(0, 5000, n)
salary = np.clip(salary, 35000, 300000).round(0)

df = pd.DataFrame({
    'EmployeeID': [f'E{i:05d}' for i in range(n)],
    'Department': departments, 'Level': levels, 'Gender': gender,
    'TenureYears': tenure_years, 'PerformanceRating': perf,
    'Education': education, 'Salary': salary,
})
print(f'Dataset: {df.shape}')
print(f'Salary stats:\n{df["Salary"].describe().round(0)}')
df.head()

## Data Validation Checks

In [ ]:
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'\nGender distribution:\n{df["Gender"].value_counts()}')
print(f'\nLevel distribution:\n{df["Level"].value_counts()}')

## Overall Salary Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(df['Salary'], bins=40, edgecolor='black', color='steelblue', alpha=0.7)
axes[0].set_title('Salary Distribution')
axes[0].set_xlabel('Salary ($)')
sns.boxplot(data=df, x='Level', y='Salary', order=['Junior','Mid','Senior','Lead','Director'], ax=axes[1])
axes[1].set_title('Salary by Level')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'salary_distribution.png'), dpi=100, bbox_inches='tight')
plt.show()

## Gender Pay Gap — Raw

In [ ]:
gender_salary = df.groupby('Gender')['Salary'].agg(['median', 'mean', 'count']).round(0)
print('Salary by Gender (raw):')
print(gender_salary)

male_median = gender_salary.loc['Male', 'median']
print(f'\nRaw gender pay gap (Female vs Male median): {(1 - gender_salary.loc["Female", "median"] / male_median):.1%}')

fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=df, x='Gender', y='Salary', ax=ax)
ax.set_title('Salary Distribution by Gender')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'gender_pay_raw.png'), dpi=100, bbox_inches='tight')
plt.show()

## Gender Pay Gap — Controlled by Level

In [ ]:
controlled = df.groupby(['Level', 'Gender'])['Salary'].median().unstack().round(0)
controlled = controlled.reindex(['Junior', 'Mid', 'Senior', 'Lead', 'Director'])
print('Median Salary by Level × Gender:')
print(controlled)

fig, ax = plt.subplots(figsize=(10, 6))
controlled.plot.bar(ax=ax, edgecolor='black')
ax.set_title('Median Salary by Level × Gender')
ax.set_ylabel('Salary ($)')
ax.tick_params(axis='x', rotation=0)
ax.legend(title='Gender')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'gender_pay_by_level.png'), dpi=100, bbox_inches='tight')
plt.show()

## Department Pay Comparison

In [ ]:
dept_stats = df.groupby('Department')['Salary'].agg(['median', 'mean', 'std', 'count']).round(0)
dept_stats = dept_stats.sort_values('median', ascending=False)
print('Salary by Department:')
print(dept_stats)

fig, ax = plt.subplots(figsize=(10, 6))
sns.boxplot(data=df, x='Department', y='Salary', ax=ax,
            order=dept_stats.index)
ax.set_title('Salary Distribution by Department')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'department_salary.png'), dpi=100, bbox_inches='tight')
plt.show()

## Tenure vs Salary Relationship

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
for g in ['Male', 'Female']:
    sub = df[df['Gender'] == g]
    ax.scatter(sub['TenureYears'], sub['Salary'], alpha=0.2, s=10, label=g)
ax.set_xlabel('Tenure (Years)')
ax.set_ylabel('Salary ($)')
ax.set_title('Salary vs Tenure by Gender')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'tenure_salary.png'), dpi=100, bbox_inches='tight')
plt.show()

corr = df[['TenureYears', 'Salary', 'PerformanceRating']].corr().round(3)
print('Correlation Matrix:')
print(corr)

## Compa-Ratio Analysis

In [ ]:
# Compa-ratio: actual salary / median salary for same Level × Department
median_ref = df.groupby(['Level', 'Department'])['Salary'].median()
df['CompaRatio'] = df.apply(lambda r: r['Salary'] / median_ref.get((r['Level'], r['Department']), r['Salary']), axis=1)

compa_by_gender = df.groupby('Gender')['CompaRatio'].agg(['mean', 'median', 'std']).round(3)
print('Compa-Ratio by Gender (1.0 = at median for role/dept):')
print(compa_by_gender)

fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=df, x='Gender', y='CompaRatio', ax=ax)
ax.axhline(1.0, color='red', linestyle='--', alpha=0.5, label='Market Median')
ax.set_title('Compa-Ratio by Gender')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'compa_ratio.png'), dpi=100, bbox_inches='tight')
plt.show()

## Education Impact on Salary

In [ ]:
edu_order = ['High School', 'Bachelors', 'Masters', 'PhD']
edu_salary = df.groupby('Education')['Salary'].median().reindex(edu_order)
print('Median Salary by Education:')
print(edu_salary.round(0))

fig, ax = plt.subplots(figsize=(8, 5))
edu_salary.plot.bar(ax=ax, edgecolor='black', color='teal')
ax.set_title('Median Salary by Education Level')
ax.set_ylabel('Salary ($)')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'education_salary.png'), dpi=100, bbox_inches='tight')
plt.show()

## Interpretation and Business Insight
- A **raw gender pay gap** exists in the data, but much of it is explained by level and department distribution
- **Controlled analysis** (same level × department) reveals a smaller but persistent gap, suggesting possible inequities in starting salary or raise practices
- **Compa-ratio** analysis confirms that Female employees cluster slightly below 1.0 more often
- **Engineering and Finance** pay premiums are significant — cross-department comparisons without role context are misleading
- **Tenure-salary correlation** is positive but moderate — promotion timing matters more than tenure alone
- **Education** matters but less than level and department — a PhD premium exists but varies by role

## Limitations
- Synthetic data with an intentionally embedded gap
- No benefits, equity compensation, or total rewards data
- No geographic cost-of-living adjustments
- Small Non-Binary sample limits statistical reliability
- No intersectional analysis (gender × race × disability)

## How to Improve This Project
- Use real anonymized compensation data
- Add regression-based pay equity modeling (controlling for multiple factors simultaneously)
- Include total compensation (bonus, equity, benefits)
- Add geographic cost-of-living normalization
- Perform intersectional analysis across multiple protected classes

## Production Considerations
- Annual pay equity audit embedded in compensation review cycle
- Automated flagging of employees significantly below compa-ratio thresholds
- Manager dashboards showing team pay equity metrics
- Legal review integration for flagged disparities

## Common Mistakes
- Reporting raw pay gaps without controlling for role, level, and tenure
- Using mean salary instead of median (outliers skew averages)
- Comparing small sample groups without noting statistical limitations
- Not distinguishing between starting salary gaps and gap accumulation over time

## Mini Challenge / Exercises
1. Calculate the pay gap controlling for Level × Department × Tenure (binned). Does a gap persist?
2. Which department has the largest within-level gender pay gap?
3. If you raised all below-median Female compa-ratios to 1.0, what would be the total budget impact?
4. Identify the 10 employees with the lowest compa-ratios. What patterns do you see?

## Final Summary / Key Takeaways
- Pay equity analysis requires controlled comparisons — raw gaps are misleading
- Compa-ratio is a practical equity metric that normalizes for role and market
- Even small persistent gaps compound over careers and must be addressed
- Regular automated equity audits are essential for fair compensation practices
- Data-driven pay equity analysis protects both employees and organizations